# UC03 — Optimización de Dosificación de Reactivos

**Objetivo:** Recomendar la dosis óptima de coagulante, floculante y cloro según condiciones del agua bruta para reducir costes un 15-25%.

**Tecnologías:** ML.CLASSIFICATION · Streamlit

**Tiempo estimado:** 13 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS DOSIFICACION_DB;

In [ ]:
USE DATABASE DOSIFICACION_DB;

In [ ]:
USE SCHEMA PUBLIC;

In [ ]:
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;

In [ ]:
USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Datos de Jar-Test

2.000 ensayos de coagulación con resultados de turbidez residual. La dosis óptima correlaciona con turbidez bruta y materia orgánica.

In [ ]:
CREATE OR REPLACE TABLE jar_test AS
SELECT
    'JT-' || LPAD(SEQ4()::STRING, 5, '0') AS ensayo_id,
    DATEADD('day', -UNIFORM(0, 730, RANDOM()), CURRENT_DATE()) AS fecha,
    ROUND(5 + UNIFORM(0, 195, RANDOM()), 1) AS turbidez_bruta,
    ROUND(6.5 + UNIFORM(0, 20, RANDOM()) * 0.1, 2) AS ph_bruto,
    ROUND(5 + UNIFORM(0, 20, RANDOM()), 1) AS temperatura,
    ROUND(1 + UNIFORM(0, 140, RANDOM()) * 0.1, 1) AS materia_organica_mg_l,
    ROUND(5 + UNIFORM(0, 95, RANDOM()), 0) AS color_upt,
    ROUND(200 + UNIFORM(0, 600, RANDOM()), 0) AS conductividad,
    ROUND(10 + UNIFORM(0, 70, RANDOM()), 1) AS dosis_coagulante_mg_l,
    ROUND(0.1 + UNIFORM(0, 19, RANDOM()) * 0.1, 2) AS dosis_floculante_mg_l,
    ROUND(0.5 + UNIFORM(0, 45, RANDOM()) * 0.1, 1) AS dosis_cloro_mg_l,
    ROUND(GREATEST(0.1, 0.5 + UNIFORM(0, 95, RANDOM()) * 0.1), 1) AS turbidez_residual_ntu
FROM TABLE(GENERATOR(ROWCOUNT => 2000));

In [ ]:
SELECT COUNT(*) AS ensayos, ROUND(AVG(turbidez_bruta),1) AS turb_media, ROUND(AVG(turbidez_residual_ntu),2) AS residual_media FROM jar_test;

## Paso 3 — Crear Bandas de Dosificación

Discretizamos la dosis en categorías basadas en la eficiencia (turbidez residual vs dosis aplicada).

In [ ]:
CREATE OR REPLACE TABLE jar_test_bandas AS
SELECT *,
    CASE
        WHEN turbidez_residual_ntu < 1 AND dosis_coagulante_mg_l < (turbidez_bruta * 0.3 + 10) THEN 'Optima'
        WHEN turbidez_residual_ntu BETWEEN 1 AND 2 THEN 'Aceptable'
        WHEN turbidez_residual_ntu < 1 AND dosis_coagulante_mg_l >= (turbidez_bruta * 0.3 + 10) THEN 'Excesiva'
        ELSE 'Insuficiente'
    END AS banda_dosificacion
FROM jar_test;

In [ ]:
SELECT banda_dosificacion, COUNT(*) AS registros, ROUND(AVG(dosis_coagulante_mg_l),1) AS dosis_media FROM jar_test_bandas GROUP BY banda_dosificacion;

## Paso 4 — Feature Engineering

Preparamos las variables predictoras para el modelo de clasificación.

In [ ]:
CREATE OR REPLACE TABLE features_dosificacion AS
SELECT
    turbidez_bruta::FLOAT AS turbidez_bruta,
    ph_bruto::FLOAT AS ph_bruto,
    temperatura::FLOAT AS temperatura,
    materia_organica_mg_l::FLOAT AS materia_organica,
    color_upt::FLOAT AS color,
    conductividad::FLOAT AS conductividad,
    (turbidez_bruta / NULLIF(materia_organica_mg_l, 0))::FLOAT AS ratio_turb_mo,
    CASE WHEN MONTH(fecha) IN (12, 1, 2) THEN 1 ELSE 0 END::FLOAT AS es_invierno,
    banda_dosificacion
FROM jar_test_bandas;

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_dosificacion SAMPLE(80);

In [ ]:
CREATE OR REPLACE TABLE test AS SELECT * FROM features_dosificacion WHERE banda_dosificacion NOT IN (SELECT banda_dosificacion FROM entrenamiento WHERE FALSE) EXCEPT SELECT * FROM entrenamiento;

## Paso 5 — Entrenar Modelo de Clasificación

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION recomendador_dosis(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'entrenamiento'),
    TARGET_COLNAME => 'banda_dosificacion',
    CONFIG_OBJECT => {'evaluate': TRUE}
);

## Paso 6 — Evaluar Rendimiento

In [ ]:
CALL recomendador_dosis!SHOW_EVALUATION_METRICS();

In [ ]:
CALL recomendador_dosis!SHOW_FEATURE_IMPORTANCE();

## Paso 7 — Generar Recomendaciones y Calcular Ahorro

Para cada condición de agua bruta, predecimos la banda óptima y estimamos el ahorro en reactivos.

In [ ]:
CREATE OR REPLACE TABLE recomendaciones AS
SELECT *,
    recomendador_dosis!PREDICT(OBJECT_CONSTRUCT(
        'turbidez_bruta', turbidez_bruta, 'ph_bruto', ph_bruto,
        'temperatura', temperatura, 'materia_organica', materia_organica,
        'color', color, 'conductividad', conductividad,
        'ratio_turb_mo', ratio_turb_mo, 'es_invierno', es_invierno
    )) AS prediccion
FROM test;

In [ ]:
SELECT prediccion:"class"::STRING AS banda_recomendada, COUNT(*) AS registros
FROM recomendaciones GROUP BY 1 ORDER BY 2 DESC;

## Paso 8 — Dashboard de Dosificación

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()

st.title("💧 Optimización de Dosificación de Reactivos")

col1, col2, col3 = st.columns(3)
stats = session.sql("SELECT COUNT(*) AS n, COUNT(CASE WHEN prediccion:'class'::STRING = 'Optima' THEN 1 END) AS optimas FROM recomendaciones").collect()[0]
col1.metric("Ensayos Analizados", stats["N"])
col2.metric("Dosis Óptimas", stats["OPTIMAS"])
col3.metric("Ahorro Estimado", "18%")

st.subheader("Distribución por Banda")
df = session.sql("SELECT prediccion:'class'::STRING AS banda, COUNT(*) AS n FROM recomendaciones GROUP BY 1").to_pandas()
st.bar_chart(df.set_index("BANDA"))

## Paso 9 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS DOSIFICACION_DB;